# Hugging Face builder test

In [9]:
# Get the current script directory, from the notebook
import os
notebook_dir = os.getcwd()
print("Notebook directory:", notebook_dir)

model_filename = "v7-2B9-world"
model_file = os.path.join(notebook_dir, ".model", f"{model_filename}.pth")
print("Model file path:", model_file)

# Check if the model file exists
if os.path.isfile(model_file) is False:
    raise Exception("Model file does not exist")

# Get the project directory two levels up
project_dir = os.path.dirname(os.path.dirname(notebook_dir))
print("Project directory:", project_dir)

# Output build directory
output_dir = os.path.join(notebook_dir, f".hf_build/{model_filename}/")
print("Output directory:", output_dir)

Notebook directory: /home/recursal/rwkv-prj/RWKV-block/test/v7_goose
Model file path: /home/recursal/rwkv-prj/RWKV-block/test/v7_goose/.model/v7-2B9-world.pth
Project directory: /home/recursal/rwkv-prj/RWKV-block
Output directory: /home/recursal/rwkv-prj/RWKV-block/test/v7_goose/.hf_build/v7-2B9-world/


In [ ]:
# Empty the output directory, if it exists
if os.path.isdir(output_dir):
    import shutil
    print("Removing existing output directory")
    shutil.rmtree(output_dir)

# Run the hf_builder.py
!python3 "$project_dir/hf_builder/hf_builder.py" --model_class "v7_goose" "$model_file" "$output_dir"

# # Run, and update only the model code (useful while debugging)
# !python3 "$project_dir/hf_builder/hf_builder.py" --model-code-only --model_class "v7_goose" "$model_file" "$output_dir"

-----------------------------
Converting RWKV model to HuggingFace format...
Model Class     : v7_goose
Model Source    : /home/recursal/rwkv-prj/RWKV-block/test/v7_goose/.model/v7-2B9-world.pth
Tokenizer Type  : auto
Output Directory: /home/recursal/rwkv-prj/RWKV-block/test/v7_goose/.hf_build/v7-2B9-world/
-----------------------------
Building rwkv_block into HF code ...
Traceback (most recent call last):
  File "/home/recursal/rwkv-prj/RWKV-block/hf_builder/hf_builder.py", line 526, in <module>
    main()
  File "/home/recursal/rwkv-prj/RWKV-block/hf_builder/hf_builder.py", line 523, in main
    hf_builder(args)
  File "/home/recursal/rwkv-prj/RWKV-block/hf_builder/hf_builder.py", line 322, in hf_builder
    build_v7_goose()
  File "/home/recursal/rwkv-prj/RWKV-block/hf_builder/hf_builder.py", line 91, in build_v7_goose
    hf_script_builder(
  File "/home/recursal/rwkv-prj/RWKV-block/hf_builder/hf_builder.py", line 180, in hf_script_builder
    if file.startswith(target_dir+"/"):
 

# Basic HELLO WORLD

In [9]:
# Load the built model, using the transformers library
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

# Validating the config and tokenizer are built correctly
config = AutoConfig.from_pretrained(output_dir, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(output_dir, trust_remote_code=True)

# Move the model to the GPU
RUN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Build the model itself
model = AutoModelForCausalLM.from_pretrained(output_dir, trust_remote_code=True, tmix_backend="triton", device=RUN_DEVICE)
model.to(RUN_DEVICE)
print("Model and tokenizer loaded successfully")

# Print the device being used
print("Running on device:", RUN_DEVICE)

# Lets generate some text, using the model on the GPU
dragon_prompt = "\nIn a shocking finding, scientist discovered a herd of dragons living in a remote, previously unexplored valley, in Tibet. Even more surprising to the researchers was the fact that the dragons spoke perfect Chinese."
hellow_prompt = "HELLO WORLD"

print("---------------------------------")
print(f"Prompt: {hellow_prompt}")
inputs = tokenizer(hellow_prompt, return_tensors="pt").to(RUN_DEVICE)
outputs = model.generate(**inputs)
print("Generated text:", tokenizer.decode(outputs[0], skip_special_tokens=True))
print("---------------------------------")
print(f"Prompt: {dragon_prompt}")
inputs = tokenizer(dragon_prompt, return_tensors="pt").to(RUN_DEVICE)
outputs = model.generate(**inputs)
print("Generated text:", tokenizer.decode(outputs[0], skip_special_tokens=True))
print("---------------------------------")

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Model and tokenizer loaded successfully
Running on device: cuda
---------------------------------
Prompt: HELLO WORLD


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Generated text: HELLO WORLD!
I am a newbie to this forum. I am trying to learn how to use the
---------------------------------
Prompt: 
In a shocking finding, scientist discovered a herd of dragons living in a remote, previously unexplored valley, in Tibet. Even more surprising to the researchers was the fact that the dragons spoke perfect Chinese.
Generated text: 
In a shocking finding, scientist discovered a herd of dragons living in a remote, previously unexplored valley, in Tibet. Even more surprising to the researchers was the fact that the dragons spoke perfect Chinese.
The dragons were discovered by a team of scientists led by Dr. John Smith, who was studying
---------------------------------


# MMLU validation testing (smaller set)
**(this is not a substitute for lm-eval-harness : the score is counted differently)**

In [4]:
# MMLU tester directory
mmlu_test_dir = os.path.join(project_dir, "test/mmlu")

# Run the test dataset builder, optional:  --use_validation_set
!python3 {mmlu_test_dir}/BuildTestMMLU.py --hf_model "$output_dir" --n_shot 0 --use_validation_set

## Using HF model tokenizer: /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/v7_goose/.hf_build/v7-1B5-world/
## Loading MMLU cached dataset (n_shot=0,tokenizer=world): /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/mmlu/.mmlu_cache/mmlu-val-t_world-n_0-p_0-c_16-r0.pth
## Done: Dataset has been built and cached


In [5]:
# Run the HF based MMLU tester, with the cuda kernel
# Batch size of 24, is for a 1B5 model, n_shot 0, with 24GB vram (ie. 4090)
!python3 {mmlu_test_dir}/RunTestMMLU.py "$output_dir" --batch_size 24 --n_shot 0 --use_validation_set --tmix_backend "cuda"

------------------------------------------------
## Loading HF model: /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/v7_goose/.hf_build/v7-1B5-world/
------------------------------------------------
## Preparing the dataset
## Loading MMLU cached dataset (n_shot=0,tokenizer=world): /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/mmlu/.mmlu_cache/mmlu-val-t_world-n_0-p_0-c_16-r0.pth
## Done: Dataset has been built and cached
------------------------------------------------
## Starting the MMLU test ...
### Running MMLU test : all (count=1531, batches=48) ...
Using /home/recursal/.cache/torch_extensions/py312_cu121 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/recursal/.cache/torch_extensions/py312_cu121/state_wind_backstepping/build.ninja...
/home/recursal/miniconda3/envs/py-3-12/lib/python3.12/site-packages/torch/utils/cpp_extension.py:1964: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for

In [6]:
# Run the HF based MMLU tester, with the triton kernel
# Batch size of 24, is for a 1B5 model, n_shot 0, with 24GB vram (ie. 4090)
!python3 {mmlu_test_dir}/RunTestMMLU.py "$output_dir" --batch_size 24 --n_shot 0 --use_validation_set --tmix_backend "triton"

------------------------------------------------
## Loading HF model: /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/v7_goose/.hf_build/v7-1B5-world/
------------------------------------------------
## Preparing the dataset
## Loading MMLU cached dataset (n_shot=0,tokenizer=world): /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/mmlu/.mmlu_cache/mmlu-val-t_world-n_0-p_0-c_16-r0.pth
## Done: Dataset has been built and cached
------------------------------------------------
## Starting the MMLU test ...
### Running MMLU test : all (count=1531, batches=48) ...
#### all - accuracy=0.3063 , probability=0.2980
------------------------------------------------
### MMLU overall test result : accuracy=0.3063 , probability=0.2980
------------------------------------------------


In [16]:
# Run the HF based MMLU tester, with the triton kernel
# Batch size of 24, is for a 1B5 model, n_shot 0, with 24GB vram (ie. 4090)
!python3 {mmlu_test_dir}/RunTestMMLU.py "$output_dir" --batch_size 24 --n_shot 0 --use_validation_set --tmix_backend "triton_bighead"

------------------------------------------------
## Loading HF model: /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/v7_goose/.hf_build/v7-1B5-world/
------------------------------------------------
## Preparing the dataset
## Loading MMLU cached dataset (n_shot=0,tokenizer=world): /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/mmlu/.mmlu_cache/mmlu-val-t_world-n_0-p_0-c_16-r0.pth
## Done: Dataset has been built and cached
------------------------------------------------
## Starting the MMLU test ...
### Running MMLU test : all (count=1531, batches=64) ...
#### all - accuracy=0.3083 , probability=0.2983
------------------------------------------------
### MMLU overall test result : accuracy=0.3083 , probability=0.2983
------------------------------------------------


In [11]:
# Run the HF based MMLU tester, with the triton kernel
# Batch size of 24, is for a 1B5 model, n_shot 0, with 24GB vram (ie. 4090)
!python3 {mmlu_test_dir}/RunTestMMLU.py "$output_dir" --batch_size 24 --n_shot 0 --use_validation_set --tmix_backend "fla"

------------------------------------------------
## Loading HF model: /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/v7_goose/.hf_build/v7-1B5-world/
------------------------------------------------
## Preparing the dataset
## Loading MMLU cached dataset (n_shot=0,tokenizer=world): /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/mmlu/.mmlu_cache/mmlu-val-t_world-n_0-p_0-c_16-r0.pth
## Done: Dataset has been built and cached
------------------------------------------------
## Starting the MMLU test ...
### Running MMLU test : all (count=1531, batches=64) ...
#### all - accuracy=0.3050 , probability=0.2982
------------------------------------------------
### MMLU overall test result : accuracy=0.3050 , probability=0.2982
------------------------------------------------


In [15]:
# Run the HF based MMLU tester, with the triton kernel
# Batch size of 24, is for a 1B5 model, n_shot 0, with 24GB vram (ie. 4090)
!python3 {mmlu_test_dir}/RunTestMMLU.py "$output_dir" --batch_size 24 --n_shot 0 --use_validation_set --tmix_backend "fla_fused"

------------------------------------------------
## Loading HF model: /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/v7_goose/.hf_build/v7-1B5-world/
------------------------------------------------
## Preparing the dataset
## Loading MMLU cached dataset (n_shot=0,tokenizer=world): /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/mmlu/.mmlu_cache/mmlu-val-t_world-n_0-p_0-c_16-r0.pth
## Done: Dataset has been built and cached
------------------------------------------------
## Starting the MMLU test ...
### Running MMLU test : all (count=1531, batches=64) ...
#### all - accuracy=0.3057 , probability=0.2982
------------------------------------------------
### MMLU overall test result : accuracy=0.3057 , probability=0.2982
------------------------------------------------


# MMLU testing 
**(this is not a substitute for lm-eval-harness : the score is counted differently)**

In [21]:
# Run the HF based MMLU tester, with the cuda kernel
# Batch size of 24, is for a 1B5 model, n_shot 0, with 24GB vram (ie. 4090)
!python3 {mmlu_test_dir}/RunTestMMLU.py "$output_dir" --batch_size 24 --n_shot 0 --tmix_backend "cuda"

------------------------------------------------
## Loading HF model: /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/v7_goose/.hf_build/v7-1B5-world/
------------------------------------------------
## Preparing the dataset
## Loading MMLU cached dataset (n_shot=0,tokenizer=world): /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/mmlu/.mmlu_cache/mmlu-test-t_world-n_0-p_0-c_16-r0.pth
## Done: Dataset has been built and cached
------------------------------------------------
## Starting the MMLU test ...
### Running MMLU test : abstract_algebra (count=100, batches=5) ...
Using /home/recursal/.cache/torch_extensions/py312_cu121 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/recursal/.cache/torch_extensions/py312_cu121/state_wind_backstepping/build.ninja...
/home/recursal/miniconda3/envs/py-3-12/lib/python3.12/site-packages/torch/utils/cpp_extension.py:1964: UserWarning: TORCH_CUDA_ARCH_LIST is not set, a

In [22]:
# Run the HF based MMLU tester, with the triton kernel (modified)
# Batch size of 24, is for a 1B5 model, n_shot 0, with 24GB vram (ie. 4090)
!python3 {mmlu_test_dir}/RunTestMMLU.py "$output_dir" --batch_size 24 --n_shot 0 --tmix_backend "triton"

------------------------------------------------
## Loading HF model: /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/v7_goose/.hf_build/v7-1B5-world/
------------------------------------------------
## Preparing the dataset
## Loading MMLU cached dataset (n_shot=0,tokenizer=world): /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/mmlu/.mmlu_cache/mmlu-test-t_world-n_0-p_0-c_16-r0.pth
## Done: Dataset has been built and cached
------------------------------------------------
## Starting the MMLU test ...
### Running MMLU test : abstract_algebra (count=100, batches=5) ...
#### abstract_algebra - accuracy=0.2100 , probability=0.2306
### Running MMLU test : anatomy (count=135, batches=6) ...
#### anatomy - accuracy=0.2963 , probability=0.3044
### Running MMLU test : astronomy (count=152, batches=7) ...
#### astronomy - accuracy=0.3882 , probability=0.3348
### Running MMLU test : business_ethics (count=100, batches=5) ...
#### business_ethics - accuracy=0.280

In [23]:
# Run the HF based MMLU tester, with the triton kernel (modified)
# Batch size of 24, is for a 1B5 model, n_shot 0, with 24GB vram (ie. 4090)
!python3 {mmlu_test_dir}/RunTestMMLU.py "$output_dir" --batch_size 24 --n_shot 0 --tmix_backend "triton_bighead"

------------------------------------------------
## Loading HF model: /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/v7_goose/.hf_build/v7-1B5-world/
------------------------------------------------
## Preparing the dataset
## Loading MMLU cached dataset (n_shot=0,tokenizer=world): /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/mmlu/.mmlu_cache/mmlu-test-t_world-n_0-p_0-c_16-r0.pth
## Done: Dataset has been built and cached
------------------------------------------------
## Starting the MMLU test ...
### Running MMLU test : abstract_algebra (count=100, batches=5) ...
#### abstract_algebra - accuracy=0.2100 , probability=0.2307
### Running MMLU test : anatomy (count=135, batches=6) ...
#### anatomy - accuracy=0.2963 , probability=0.3042
### Running MMLU test : astronomy (count=152, batches=7) ...
#### astronomy - accuracy=0.3882 , probability=0.3343
### Running MMLU test : business_ethics (count=100, batches=5) ...
#### business_ethics - accuracy=0.280

In [24]:
# Run the HF based MMLU tester, with the triton kernel (modified)
# Batch size of 24, is for a 1B5 model, n_shot 0, with 24GB vram (ie. 4090)
!python3 {mmlu_test_dir}/RunTestMMLU.py "$output_dir" --batch_size 24 --n_shot 0 --tmix_backend "fla"

------------------------------------------------
## Loading HF model: /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/v7_goose/.hf_build/v7-1B5-world/
------------------------------------------------
## Preparing the dataset
## Loading MMLU cached dataset (n_shot=0,tokenizer=world): /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/mmlu/.mmlu_cache/mmlu-test-t_world-n_0-p_0-c_16-r0.pth
## Done: Dataset has been built and cached
------------------------------------------------
## Starting the MMLU test ...
### Running MMLU test : abstract_algebra (count=100, batches=5) ...
#### abstract_algebra - accuracy=0.2100 , probability=0.2309
### Running MMLU test : anatomy (count=135, batches=6) ...
#### anatomy - accuracy=0.2963 , probability=0.3048
### Running MMLU test : astronomy (count=152, batches=7) ...
#### astronomy - accuracy=0.3882 , probability=0.3346
### Running MMLU test : business_ethics (count=100, batches=5) ...
#### business_ethics - accuracy=0.280

In [25]:
# Run the HF based MMLU tester, with the triton kernel (modified)
# Batch size of 32, is for a 1B5 model, n_shot 0, with 24GB vram (ie. 4090)
!python3 {mmlu_test_dir}/RunTestMMLU.py "$output_dir" --batch_size 24 --n_shot 0 --tmix_backend "fla_fused"

------------------------------------------------
## Loading HF model: /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/v7_goose/.hf_build/v7-1B5-world/
------------------------------------------------
## Preparing the dataset
## Loading MMLU cached dataset (n_shot=0,tokenizer=world): /home/recursal/rwkv-prj/layerwise-trainer/block/RWKV_block/test/mmlu/.mmlu_cache/mmlu-test-t_world-n_0-p_0-c_16-r0.pth
## Done: Dataset has been built and cached
------------------------------------------------
## Starting the MMLU test ...
### Running MMLU test : abstract_algebra (count=100, batches=5) ...
#### abstract_algebra - accuracy=0.2100 , probability=0.2309
### Running MMLU test : anatomy (count=135, batches=6) ...
#### anatomy - accuracy=0.2963 , probability=0.3040
### Running MMLU test : astronomy (count=152, batches=7) ...
#### astronomy - accuracy=0.3882 , probability=0.3345
### Running MMLU test : business_ethics (count=100, batches=5) ...
#### business_ethics - accuracy=0.280

# LM Eval harness testing
The real MMLU test

In [3]:
# Test the base model
!NCCL_IB_DISABLE=1 NCCL_P2P_DISABLE=1 lm_eval --model hf \
    --model_args pretrained="$output_dir",dtype="bfloat16",trust_remote_code=True,tmix_backend="cuda" \
    --tasks mmlu \
    --device "cuda:6" \
    --batch_size 8 # This is adjusted for a 24GB system

2025-03-16:07:13:44,841 INFO     [lm_eval.__main__:379] Selected Tasks: ['mmlu']
2025-03-16:07:13:44,843 INFO     [lm_eval.evaluator:177] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-03-16:07:13:44,843 INFO     [lm_eval.evaluator:214] Initializing hf model, with arguments: {'pretrained': '/home/recursal/rwkv-prj/RWKV-block/test/v7_goose/.hf_build/v7-2B9-world/', 'dtype': 'bfloat16', 'trust_remote_code': True, 'tmix_backend': 'cuda'}
2025-03-16:07:13:45,071 INFO     [lm_eval.models.huggingface:136] Using device 'cuda:6'
Traceback (most recent call last):
  File "/home/recursal/miniconda3/envs/py-3-12/lib/python3.12/site-packages/transformers/utils/hub.py", line 403, in cached_file
    resolved_file = hf_hub_download(
                    ^^^^^^^^^^^^^^^^
  File "/home/recursal/miniconda3/envs/py-3-12/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py", line 106, in _inner_fn
    validat